In [7]:
"""
continuous_maze.py

Maze in continuous 2D space where walls are line segments.
From the gym-continuous-maze repo with RL parts removed.
"""

import numpy as np
from typing import Optional

from shapely.geometry import LineString, MultiLineString, Point
from shapely.prepared import prep

def get_intersect(A: np.ndarray, B: np.ndarray, C: np.ndarray, D: np.ndarray) -> Optional[np.ndarray]:
    """
    Get the intersection of segment [A, B] and segment [C, D].
    Return the intersection point if the segments cross, otherwise None.

    :param A: First point of the first segment
    :param B: Second point of the first segment
    :param C: First point of the second segment
    :param D: Second point of the second segment
    :return: The intersection point [x, y] if they cross, otherwise None.
    """
    # A, B, C, D = np.asarray(A), np.asarray(B), np.asarray(C), np.asarray(D)
    det = (B[0] - A[0]) * (C[1] - D[1]) - (C[0] - D[0]) * (B[1] - A[1])
    if det == 0:
        return None
    else:
        t1 = ((C[0] - A[0]) * (C[1] - D[1]) - (C[0] - D[0]) * (C[1] - A[1])) / det
        t2 = ((B[0] - A[0]) * (C[1] - A[1]) - (C[0] - A[0]) * (B[1] - A[1])) / det

        if t1 <= 0 or t1 >= 1 or t2 <= 0 or t2 >= 1:
            return None
        else:
            xi = A[0] + t1 * (B[0] - A[0])
            yi = A[1] + t1 * (B[1] - A[1])
            return np.array([xi, yi])


# Maze walls: list of segments. Each segment is [[x1,y1], [x2,y2]].
# Bounds of this maze: x and y in [-12, 12].
walls = np.array(
    [
        [[-12.0, -12.0], [-12.0, 12.0]],
        [[-10.0, 8.0], [-10.0, 10.0]],
        [[-10.0, 0.0], [-10.0, 6.0]],
        [[-10.0, -4.0], [-10.0, -2.0]],
        [[-10.0, -10.0], [-10.0, -6.0]],
        [[-8.0, 4.0], [-8.0, 8.0]],
        [[-8.0, -4.0], [-8.0, 0.0]],
        [[-8.0, -8.0], [-8.0, -6.0]],
        [[-6.0, 8.0], [-6.0, 10.0]],
        [[-6.0, 4.0], [-6.0, 6.0]],
        [[-6.0, 0.0], [-6.0, 2.0]],
        [[-6.0, -6.0], [-6.0, -4.0]],
        [[-4.0, 2.0], [-4.0, 8.0]],
        [[-4.0, -2.0], [-4.0, 0.0]],
        [[-4.0, -10.0], [-4.0, -6.0]],
        [[-2.0, 8.0], [-2.0, 12.0]],
        [[-2.0, 2.0], [-2.0, 6.0]],
        [[-2.0, -4.0], [-2.0, -2.0]],
        [[0.0, 6.0], [0.0, 12.0]],
        [[0.0, 2.0], [0.0, 4.0]],
        [[0.0, -8.0], [0.0, -6.0]],
        [[2.0, 8.0], [2.0, 10.0]],
        [[2.0, -8.0], [2.0, 6.0]],
        [[4.0, 10.0], [4.0, 12.0]],
        [[4.0, 4.0], [4.0, 6.0]],
        [[4.0, 0.0], [4.0, 2.0]],
        [[4.0, -6.0], [4.0, -2.0]],
        [[4.0, -10.0], [4.0, -8.0]],
        [[6.0, 10.0], [6.0, 12.0]],
        [[6.0, 6.0], [6.0, 8.0]],
        [[6.0, 0.0], [6.0, 2.0]],
        [[6.0, -8.0], [6.0, -6.0]],
        [[8.0, 10.0], [8.0, 12.0]],
        [[8.0, 4.0], [8.0, 6.0]],
        [[8.0, -4.0], [8.0, 2.0]],
        [[8.0, -10.0], [8.0, -8.0]],
        [[10.0, 10.0], [10.0, 12.0]],
        [[10.0, 4.0], [10.0, 8.0]],
        [[10.0, -2.0], [10.0, 0.0]],
        [[12.0, -12.0], [12.0, 12.0]],
        [[-12.0, 12.0], [12.0, 12.0]],
        [[-12.0, 10.0], [-10.0, 10.0]],
        [[-8.0, 10.0], [-6.0, 10.0]],
        [[-4.0, 10.0], [-2.0, 10.0]],
        [[2.0, 10.0], [4.0, 10.0]],
        [[-8.0, 8.0], [-2.0, 8.0]],
        [[2.0, 8.0], [8.0, 8.0]],
        [[-10.0, 6.0], [-8.0, 6.0]],
        [[-6.0, 6.0], [-2.0, 6.0]],
        [[6.0, 6.0], [8.0, 6.0]],
        [[0.0, 4.0], [6.0, 4.0]],
        [[-10.0, 2.0], [-6.0, 2.0]],
        [[-2.0, 2.0], [0.0, 2.0]],
        [[8.0, 2.0], [10.0, 2.0]],
        [[-4.0, 0.0], [-2.0, 0.0]],
        [[2.0, 0.0], [4.0, 0.0]],
        [[6.0, 0.0], [8.0, 0.0]],
        [[-6.0, -2.0], [2.0, -2.0]],
        [[4.0, -2.0], [10.0, -2.0]],
        [[-12.0, -4.0], [-8.0, -4.0]],
        [[-4.0, -4.0], [-2.0, -4.0]],
        [[0.0, -4.0], [6.0, -4.0]],
        [[8.0, -4.0], [10.0, -4.0]],
        [[-8.0, -6.0], [-6.0, -6.0]],
        [[-2.0, -6.0], [0.0, -6.0]],
        [[6.0, -6.0], [10.0, -6.0]],
        [[-12.0, -8.0], [-6.0, -8.0]],
        [[-2.0, -8.0], [2.0, -8.0]],
        [[4.0, -8.0], [6.0, -8.0]],
        [[8.0, -8.0], [10.0, -8.0]],
        [[-10.0, -10.0], [-8.0, -10.0]],
        [[-4.0, -10.0], [4.0, -10.0]],
        [[-12.0, -12.0], [12.0, -12.0]],
    ]
)

# Fast way of checking if an edge will intersect a line segment in the maze
obstacles = prep(MultiLineString([[(w[0][0], w[0][1]), (w[1][0], w[1][1])] for w in walls]))

In [ ]:
# pip install shapely

  Using cached shapely-2.1.2-cp312-cp312-macosx_11_0_arm64.whl.metadata (6.8 kB)
Using cached shapely-2.1.2-cp312-cp312-macosx_11_0_arm64.whl (1.6 MB)

[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [8]:
p=Point(8,8)
obstacles.context.disjoint(p)

False

In [9]:
obstacles.context.distance(p)

0.0

In [10]:
p=Point(9,8)
obstacles.context.distance(p)

1.0

In [13]:
L=LineString([(9,8),(9,9)])
obstacles.context.distance(L)

1.0